## E0

### modelacion de entidades

Tip: ver bien el archivo, entender qué está pasando, y así darse una idea de qué se puede buscar.

In [18]:
class Actor:
    def __init__(self, nombre): # todos los parámetros que tienen que pasarle a la clase para ser definida.
        self.nombre = nombre
        self.peliculas = []

    def agregar_pelicula(self, pelicula):
        self.peliculas.append(pelicula)
    
class Genero:
    def __init__(self, nombre):
        self.nombre = nombre
        self.peliculas = []

    def agregar_pelicula(self, pelicula):
        self.peliculas.append(pelicula)

class Pelicula:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio
        self.generos = []
        self.actores = []

    def agregar_actor(self, actor: Actor):
        self.actores.append(actor)
        actor.agregar_pelicula(self)

    def agregar_genero(self, genero):
        self.generos.append(genero)
        genero.agregar_pelicula(self)

    # def __str__(self):
    #     return f"{self.titulo} ({self.anio})" # investigar esto.

### carga de datos

In [19]:
import json

with open('movies.json', encoding='utf8') as f:
    data = json.load(f)

In [20]:
print(f"Se han cargado {len(data)} películas del archivo JSON.")

Se han cargado 28795 películas del archivo JSON.


In [21]:
# imprimir 30 películas para verificar que se cargaron correctamente
for i in range(30):
    print(data[i])

{'title': 'After Dark in Central Park', 'year': 1900, 'cast': [], 'genres': []}
{'title': "Boarding School Girls' Pajama Parade", 'year': 1900, 'cast': [], 'genres': []}
{'title': "Buffalo Bill's Wild West Parad", 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Caught', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Clowns Spinning Hats', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Capture of Boer Battery by British', 'year': 1900, 'cast': [], 'genres': ['Short', 'Documentary']}
{'title': 'The Enchanted Drawing', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Feeding Sea Lions', 'year': 1900, 'cast': ['Paul Boyton'], 'genres': []}
{'title': 'How to Make a Fat Wife Out of Two Lean Ones', 'year': 1900, 'cast': [], 'genres': ['Comedy']}
{'title': 'New Life Rescue', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'New Morning Bath', 'year': 1900, 'cast': [], 'genres': []}
{'title': 'Searching Ruins on Broadway, Galveston, for Dead Bodies', 'year': 1900, 'cast': [], 'gen

In [22]:
# funcion auxiliar
def es_actor_valido(nombre):
    if not nombre:
        return False
    
    nombre = nombre.strip()

    if nombre == "":
        return False
    
    basura = {"and", ".", ",", "(voice)"}
    if nombre.lower() in basura:
        return False
    
    # descartar anotaciones entre paréntesis
    if nombre.startswith("(") and nombre.endswith(")"):
        return False
    
    # algo opcional: evitar nombres demasiado cortos, si es que hay casos de eso
    if len(nombre) < 3:
        return False
    
    return True


In [23]:
# actores_dict = {'Juan': Actor('Juan'), 'Pedro': Actor('Pedro')}  # diccionario de actores
actores_dict = {} # llave: valor.
generos_dict = {} # {'Short': Genero('Short'), 'Comedy': Genero('Comedy')}
peliculas = []

In [24]:
peliculas_vistas = set() # busquen qué es

for movie in data:
    cast = []
    generos = []

    # actores
    for a in movie["cast"]:
        if a not in actores_dict: # Juan, Pedro, ...
            actores_dict[a] = Actor(a)
        actor = actores_dict[a]
        cast.append(actor)

    # géneros
    for g in movie["genres"]:
        if g not in generos_dict:
            generos_dict[g] = Genero(g)

        genero = generos_dict[g]
        generos.append(genero)

    clave = (movie["title"], movie["year"], tuple(cast))

    if clave not in peliculas_vistas:
        pelicula = Pelicula(movie["title"], movie["year"])
        
        for genero in generos:
            pelicula.agregar_genero(genero)

        for actor in cast:
            pelicula.agregar_actor(actor)

        peliculas.append(pelicula)
        peliculas_vistas.add(clave)
    else:
        print(f"Película duplicada encontrada: {movie['title']} ({movie['year']})")

Película duplicada encontrada: The Prince and Betty (1919)
Película duplicada encontrada: The Virtuous Thief (1919)
Película duplicada encontrada: The Wise Kid (1922)
Película duplicada encontrada: The White Sister (1923)
Película duplicada encontrada: Wild Bill Hickok Rides (1942)
Película duplicada encontrada: The Rough, Tough West (1952)


In [25]:
print(f"dict_peliculas tiene {len(peliculas)} películas únicas.")

dict_peliculas tiene 28789 películas únicas.


Vemos que 28789 + 6 = 28795. Así que está bien

### Consultas sobre los datos

1) Encuentre los 5 generos mas populares.

In [26]:
# {'Short': Genero('Short'), 'Comedy': Genero('Comedy')}

In [27]:
# conteo = [(g.nombre, len(g.peliculas)) for g in generos_dict.values()]
# otra forma de contar:
conteo = []
for g in generos_dict.values():
    conteo.append((g.nombre, len(g.peliculas))) # tupla = inmutable, no se puede modificar. (g.nombre, len(g.peliculas)) es una tupla.
# usen la que les parezca más clara y eficiente

# revisar diferencia entre sort y sorted. sorted devuelve una lista nueva, mientras que sort modifica la lista original.
# hasta ahora, conteo es = [('Drama', 123), ('Comedy', 98), ('Action', 76), ...]

top_generos = sorted(conteo, key=lambda x: x[1], reverse=True)[:5]

# tip: revisar funciones lambda y sorted para agilizar el conteo y ordenamiento
for genero, count in top_generos:
    print(f"Género: {genero}, cantidad de películas: {count}")

Género: Drama, cantidad de películas: 8742
Género: Comedy, cantidad de películas: 7361
Género: Western, cantidad de películas: 3011
Género: Crime, cantidad de películas: 1499
Género: Horror, cantidad de películas: 1166


2)  Encuentre los 3 años con mas peliculas estrenadas

In [28]:
peliculas[:3]

In [29]:
conteo_anios = {} # {'1999': 123, '2000': 98, '2001': 76, ...}

for p in peliculas:
    if p.anio not in conteo_anios:
        conteo_anios[p.anio] = 0
    conteo_anios[p.anio] += 1

print(conteo_anios.items())
# ordenar
top_anios = sorted(conteo_anios.items(), key=lambda x: x[1], reverse=True)[:3]

for anio, count in top_anios:
    print(f"Año: {anio}, cantidad de películas: {count}")

dict_items([(1900, 17), (1901, 80), (1902, 7), (1903, 78), (1904, 25), (1905, 35), (1906, 7), (1907, 7), (1908, 18), (1909, 76), (1910, 26), (1911, 28), (1912, 44), (1913, 53), (1914, 179), (1915, 110), (1916, 144), (1917, 97), (1918, 128), (1919, 632), (1920, 129), (1921, 143), (1922, 420), (1923, 394), (1924, 480), (1925, 572), (1926, 491), (1927, 169), (1928, 437), (1929, 347), (1930, 361), (1931, 360), (1932, 408), (1933, 383), (1934, 407), (1935, 446), (1936, 504), (1937, 473), (1938, 423), (1939, 339), (1940, 385), (1941, 358), (1942, 358), (1943, 465), (1944, 456), (1945, 415), (1946, 423), (1947, 390), (1948, 421), (1949, 351), (1950, 443), (1951, 429), (1952, 373), (1953, 366), (1954, 251), (1955, 268), (1956, 300), (1957, 335), (1958, 281), (1959, 205), (1960, 162), (1961, 150), (1962, 144), (1963, 140), (1964, 151), (1965, 129), (1966, 131), (1967, 127), (1968, 143), (1969, 137), (1970, 137), (1971, 145), (1972, 140), (1973, 142), (1974, 142), (1975, 128), (1976, 149), (1977

3) Encuentre a los 5 actores con la trayectoria mas larga, es decir, mayor cantidad de años actuando.

In [30]:
actores_dict

{'Paul Boyton': <__main__.Actor at 0x1aede5a1000>,
 'Ching Ling Foo': <__main__.Actor at 0x1aede5a2bf0>,
 'May Clark': <__main__.Actor at 0x1aedd25a2f0>,
 'William Carrington': <__main__.Actor at 0x1aede6a37f0>,
 'J. Stuart Blackton': <__main__.Actor at 0x1aedf23ef50>,
 'Florence Lawrence': <__main__.Actor at 0x1aedf23dc00>,
 'William S. Hart': <__main__.Actor at 0x1aedf23e740>,
 'William Craven': <__main__.Actor at 0x1aedf23fa00>,
 'Unknown': <__main__.Actor at 0x1aedf23e110>,
 'Bertha Regustus': <__main__.Actor at 0x1aedf23e620>,
 'Edward Boulden': <__main__.Actor at 0x1aedf23f9a0>,
 'Arthur V. Johnson': <__main__.Actor at 0x1aedf23df60>,
 'Linda Arvidson': <__main__.Actor at 0x1aedf23f400>,
 'William V. Ranous': <__main__.Actor at 0x1aedf23da50>,
 'George Gebhardt': <__main__.Actor at 0x1aedf23ec20>,
 'Charles Inslee': <__main__.Actor at 0x1aedf23dba0>,
 'D. W. Griffith': <__main__.Actor at 0x1aedf23f220>,
 'Harry Solter': <__main__.Actor at 0x1aedf23f700>,
 'Tom Ricketts': <__main_

In [31]:
trayectoria = [] # ('Juan', 10) # ('nombre_actor', duracion_trayectoria)

for actor in actores_dict.values():
    if not es_actor_valido(actor.nombre):
        # print(f"Actor inválido descartado: '{actor.nombre}'")
        continue
    # years = [p.anio for p in actor.peliculas] # [1919, 1954, 2001]
    years = []
    for p in actor.peliculas:
        years.append(p.anio)
    if years:
        trayectoria.append((actor.nombre, max(years) - min(years)))

top_actores = sorted(trayectoria, key=lambda x: x[1], reverse=True)[:5]
for actor, duracion in top_actores:
    print(f"Actor: {actor}, duración de trayectoria: {duracion} años")

Actor: Harrison Ford, duración de trayectoria: 98 años
Actor: Gloria Stuart, duración de trayectoria: 80 años
Actor: Lillian Gish, duración de trayectoria: 75 años
Actor: Kenny Baker, duración de trayectoria: 75 años
Actor: Mickey Rooney, duración de trayectoria: 74 años


In [32]:
# imprimir pelis de Harrison Ford
harrison_ford = actores_dict.get("Harrison Ford")
print(f"Películas de Harrison Ford: {[p.titulo for p in harrison_ford.peliculas]}")

Películas de Harrison Ford: ['Experimental Marriage', 'Happiness a la Mode', 'Romance and Arabella', 'The Third Kiss', 'The Veiled Adventure', 'Who Cares?', 'You Never Saw Such a Girl', 'The Wonderful Thing', 'Find the Woman', 'The Primitive Lover', "Smilin' Through", 'When Love Comes', 'Little Old New York', 'Three Miles Out', 'The Wheel', 'Almost a Lady', "Hell's Four Hundred", 'The Nervous Wreck', "Up in Mabel's Room", 'Golf Widows', "Let 'Er Go Gallegher", 'A Woman Against the World', 'American Graffiti', 'The Conversation', 'Heroes', 'Star Wars Episode IV: A New Hope (aka Star Wars)', 'The Frisco Kid', 'Hanover Street', 'The Empire Strikes Back', 'Raiders of the Lost Ark', 'Blade Runner', 'Return of the Jedi', 'Indiana Jones and the Temple of Doom', 'Witness', 'The Mosquito Coast', 'Frantic', 'Working Girl', 'Indiana Jones and the Last Crusade', 'Presumed Innocent', 'Regarding Henry', 'Patriot Games', 'The Fugitive', 'Clear and Present Danger', 'Sabrina', 'Air Force One', 'Six Day

Acá hay un alcance de nombre, los dos se llaman igual :c, por eso salió 98 años.

Prueben descomentando la linea del actor inválido, y se darán cuenta de por qué decidí crear la función actor válido.

4)  Encuentre el reparto de una pelicula (2 o mas actores) que se haya repetido completo en otras la mayor
cantidad de veces

In [33]:
conteo_repartos = {} # {('Actor1', 'Actor2'): 3, ('Actor3', 'Actor4', 'Actor5'): 2, ...}

for p in peliculas:
    reparto_completo = []

    for a in p.actores:
        nombre = a.nombre.strip()
        if es_actor_valido(nombre):
            reparto_completo.append(nombre)

    reparto_completo = tuple(reparto_completo)

    if len(reparto_completo) >= 2:
        if reparto_completo not in conteo_repartos:
            conteo_repartos[reparto_completo] = 0
        conteo_repartos[reparto_completo] += 1

max_reparto = None
max_count = 0

for reparto, count in conteo_repartos.items():
    if count > max_count:
        max_count = count
        max_reparto = reparto

print(f"Reparto completo más repetido: {max_reparto}")
print(f"Cantidad de veces: {max_count}")

Reparto completo más repetido: ('Harold Lloyd', 'Bebe Daniels')
Cantidad de veces: 44
